In [1]:
!pip install timm opencv-python
!git clone https://github.com/isl-org/MiDaS.git
%cd MiDaS

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 47.7 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.8.93
    Uninstalling nvidia-nvjitlink-cu12-12.8.93:
      Successfully uninstalled nvidia-nvjitlink-cu12-12.8.93
  Attempting uninstall: nvidia-curand-cu12
    Found existing installation: nvidia-curand-cu12 10.3.9.90
    Uninstalling nvidia-curand-cu12-10.3.9.90:
      Successfully uninstalled nvidia-curand-cu12-10.3.9.90
  Attemptin

In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load
"""
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session
"""

'\nimport numpy as np # linear algebra\nimport pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)\n\n# Input data files are available in the read-only "../input/" directory\n# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory\n\nimport os\nfor dirname, _, filenames in os.walk(\'/kaggle/input\'):\n    for filename in filenames:\n        print(os.path.join(dirname, filename))\n\n# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" \n# You can also write temporary files to /kaggle/temp/, but they won\'t be saved outside of the current session\n'

In [3]:
import os
import torch
import cv2
import numpy as np
import glob
import random
from midas.dpt_depth import DPTDepthModel
from torchvision.transforms import Compose
from midas.transforms import Resize, NormalizeImage, PrepareForNet

/usr/local/lib/python3.11/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [4]:
# Load model
model_path = "/kaggle/input/dpt_hybrid_384/pytorch/default/1/dpt_hybrid_384.pt"
model = DPTDepthModel(
    path=model_path,
    backbone="vitb_rn50_384",
    non_negative=True,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# Transforms
transform = Compose([
    Resize(
        width=384, height=384,
        resize_target=False,
        keep_aspect_ratio=True,
        ensure_multiple_of=32,
        resize_method="minimal",
    ),
    NormalizeImage(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    PrepareForNet()
])

/usr/local/lib/python3.11/dist-packages/timm/models/_factory.py:126: UserWarning: Mapping deprecated model name vit_base_resnet50_384 to current vit_base_r50_s16_384.orig_in21k_ft_in1k.
  model = create_fn(
/kaggle/working/MiDaS/midas/base_model.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don'

In [5]:
# Folder containing training samples
train_dir = "/kaggle/input/ethz-cil-monocular-depth-estimation-2025/train/train"

# Get all RGB image file paths based on your naming convention
all_rgb_paths = glob.glob(os.path.join(train_dir, "*_rgb.png"))
# Extract base names (e.g. "sample_00000") without "_rgb.png"
base_names = [os.path.basename(path).replace("_rgb.png", "") for path in all_rgb_paths]

# Randomly sample a subset of images for calibration (e.g. 200 samples)
num_samples = 1000
sample_names = random.sample(base_names, min(num_samples, len(base_names)))


In [6]:
all_preds = []
all_gts = []

for name in sample_names:
    rgb_path = os.path.join(train_dir, f"{name}_rgb.png")
    depth_path = os.path.join(train_dir, f"{name}_depth.npy")
    
    # Load RGB image and ground truth depth
    img = cv2.imread(rgb_path)
    gt_depth = np.load(depth_path)
    
    # Apply transforms to get model input
    inp = transform({"image": img})["image"]
    inp = torch.from_numpy(inp).unsqueeze(0).to(device)
    
    # Run inference
    with torch.no_grad():
        pred_depth = model(inp).squeeze().cpu().numpy()
    
    # Resize model output back to original dimensions (width=560, height=426)
    pred_depth_resized = cv2.resize(pred_depth, (560, 426))
    
    # Flatten and store for calibration
    all_preds.append(pred_depth_resized.flatten())
    all_gts.append(gt_depth.flatten())

# Concatenate all the flattened results from each sample
preds_flat = np.concatenate(all_preds)
gts_flat = np.concatenate(all_gts)


In [7]:
# Construct matrix A for least squares: [predictions, ones]
A = np.vstack([preds_flat, np.ones_like(preds_flat)]).T
a, b = np.linalg.lstsq(A, gts_flat, rcond=None)[0]

print(f"Calibrated scale (a): {a}")
print(f"Calibrated shift (b): {b}")

Calibrated scale (a): -0.001575236557982862
Calibrated shift (b): 4.44881010055542


In [8]:
test_dir = "/kaggle/input/ethz-cil-monocular-depth-estimation-2025/test/test"
# Two output directories: one for .npy files and one for visual images.
output_npy_dir = "/kaggle/working/preds_npy"
output_img_dir = "/kaggle/working/preds_img"
os.makedirs(output_npy_dir, exist_ok=True)
os.makedirs(output_img_dir, exist_ok=True)

# Load test list.
with open("/kaggle/input/ethz-cil-monocular-depth-estimation-2025/test_list.txt", "r") as f:
    test_list = [line.strip().split()[0] for line in f]

for filename in test_list:
    img_path = os.path.join(test_dir, filename)
    img = cv2.imread(img_path)
    img_input = transform({"image": img})["image"]
    img_input = torch.from_numpy(img_input).unsqueeze(0).to(device)

    with torch.no_grad():
        depth = model(img_input).squeeze().cpu().numpy()

    # Apply calibration to convert raw depth to metric depth.
    depth = a * depth + b
    depth = np.maximum(depth, 0.1)
    # Resize depth map back to original size (560x426).
    depth = cv2.resize(depth, (560, 426))

    # Save .npy prediction.
    base = filename.replace("_rgb.png", "_depth.npy")
    save_path_npy = os.path.join(output_npy_dir, base)
    np.save(save_path_npy, depth)

    # Save a visualization image.
    # Normalize depth for visualization (0-255) using cv2.normalize.
    depth_vis = cv2.normalize(depth, None, 0, 255, cv2.NORM_MINMAX)
    depth_vis = np.uint8(depth_vis)
    # Apply a colormap.
    depth_vis = cv2.applyColorMap(depth_vis, cv2.COLORMAP_JET)
    # Save visual image.
    save_path_img = os.path.join(output_img_dir, base.replace(".npy", ".png"))
    cv2.imwrite(save_path_img, depth_vis)